# ECB USD/EUR: exploratory data analysis

We forecast the next published reference rate: EUR per one USD. Source: AllRatesToday / ECB, Kaggle version 131. Data licence: CC BY 4.0. The raw snapshot and SHA-256 are fixed in `data/manifest.json`.

In [ ]:
import os
from pathlib import Path

from IPython.display import Markdown, display

from src.config import get_settings
from src.ml.data import load_rates, read_manifest
from src.ml.eda import log_eda, make_plots, make_report

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)

rates = load_rates()
read_manifest()

## 1. Size, dates and data quality

Select only the ECB EUR→USD reference series and invert it. Reject duplicate dates and invalid values. Do not fill weekends with copies of Friday.

In [ ]:
print(f"Observations: {len(rates)}")
print(f"Dates: {rates.date.min().date()} — {rates.date.max().date()}")
print("Missing values:", rates.isna().sum().to_dict())
print("Duplicate dates:", rates.date.duplicated().sum())
rates.head()

In [ ]:
rates[["rate"]].describe()

## 2. Rate, changes and volatility

Inspect the full series for data quality. Use chronological training/validation windows for model choices; leave the final holdout for the selected candidate.

In [ ]:
figures = make_plots(rates)
for figure in figures.values():
    display(figure)

## 3. Findings and limitations

Large moves may be genuine market events; do not delete them just because they are large. Reference rates are published once per business day. They are different from continuously changing market prices. A constant last-value forecast is our baseline.

In [ ]:
display(Markdown(make_report(rates)))

## 4. Save the evidence to MLflow

The EDA Run contains the Dataset input, source, digest, original CSV, manifest, four plots and report. Dataset Tracking records metadata; we also upload the snapshot explicitly.

In [ ]:
run_id = log_eda(get_settings(), rates, figures)
print("EDA Run:", run_id)